# Pętla wykonawcza: Dispatch

## Cel: Zmienisz statyczne definicje w autonomiczny system, implementując funkcję `dispatch` i domykając pętlę sprzężenia zwrotnego, która zamienia probabilistyczne plany w deterministyczne działanie.

### Czego się nauczysz:
1.  **Architektura `NextStep`:** Zrozumiesz, dlaczego ta klasa to w rzeczywistości ukryty wzorzec **Kaskady i Routera**, który wymusza na modelu myślenie przed działaniem.
2.  **Most `dispatch`:** Zaimplementujesz jedyny punkt styku między "wyobraźnią" AI a twardym kodem (Software 1.0), co gwarantuje bezpieczeństwo i przewidywalność systemu.
3.  **Strategia Discovery:** Nauczysz się oddzielać fazę zbierania danych (Memory Tools) od fazy interakcji (Action), aby agent nie podejmował decyzji na podstawie halucynacji.
4.  **Mutowalny kontekst:** Zobaczysz, jak dynamicznie wstrzykiwać "odkrytą wiedzę" do promptu systemowego w czasie rzeczywistym, zamiast zaśmiecać historię czatu.
5.  **Zarządzanie stanem:** Dowiesz się, dlaczego wydzielona lista `discoveries` daje znacznie lepszą sterowalność procesem niż poleganie wyłącznie na pamięci krótkotrwałej modelu.
6.  **Symulacja środowiska:** Stworzysz deterministyczny "Shadow System" (symulator restauracji), który pozwoli Ci bezpiecznie testować scenariusze brzegowe (np. brak miejsc).
7.  **Domknięcie pętli:** Zepniesz wszystkie elementy w jedną całość, tworząc mechanizm, który potrafi całkowicie zmienić plan działania w połowie procesu, gdy napotka nowe fakty.

W poprzedniej lekcji (4.2) stworzyliśmy fundamenty naszego systemu: `NextStep` oraz pamięć `WorkingMemory`, `ConstitutionalRules`. Zdefiniowaliśmy, *jak* agent ma myśleć i *co* teoretycznie może zrobić. Jednak nasz `dispatch` był wtedy tylko prostym mechanizmem typu "Hello World", który obsługiwał wyłącznie stany rozmowy.

Teraz czas zrobić krok do przodu. W tej lekcji rozbudujemy logikę o dodatkowy element: narzędzia.

Zanim nasz agent podniesie słuchawkę i zadzwoni do restauracji, musi najpierw "odrobić pracę domową". Zasymulujemy dostęp do dwóch baz wiedzy (mockup):
1.  **Baza klientów (pamięć semantyczna):** Podstawowe dane o osobie zlecającej (np. preferencje, alergie).
2.  **Historia zdarzeń (pamięć epizodyczna):** Informacje o poprzednich interakcjach z daną restauracją (np. "ostatnio długo czekaliśmy na stolik").

Dopiero wyposażony w ten kontekst, agent AI będzie mógł podjąć świadomą decyzję.

W tej lekcji skupimy się na dwóch mechanizmach, które to umożliwią:

Po pierwsze: **Struktura `NextStep`**. Choć w kodzie to nadal prosta klasa z kilkoma polami, w rzeczywistości pełni ona funkcję kompletnej **mikro-architektury decyzyjnej**. Łączy w sobie wzorce, o których mówiliśmy wcześniej:
- **Kaskada:** Wymuszamy kolejność przetwarzania (najpierw analiza `current_state`, potem planowanie `plan`, na końcu decyzja `next_action`).
- **Router:** Pole `next_action` działa jak zwrotnica – model musi jednoznacznie wybrać jedną ścieżkę (użyć narzędzia do pobrania danych LUB rozpocząć rozmowę), odrzucając inne opcje.
- **Cykl:** Wynik działania (np. pobrane dane z bazy) wraca do modelu w kolejnej turze, domykając pętlę sprzężenia zwrotnego.

Po drugie: funkcja `dispatch`. To miejsce, gdzie "słowo staje się ciałem". W poprzedniej lekcji tylko "drukowaliśmy" stany. Teraz `dispatch` stanie się inteligentnym zarządcą, który potrafi nie tylko generować wypowiedzi, ale też odpytywać "bazy danych" i wstrzykiwać odnalezione informacje z powrotem do kontekstu agenta.

Zatem – bierzmy się do kodu.

## Wczytujemy biblioteki

In [3]:
from typing import List, Dict, Union, Literal, Annotated, Optional, Any
from enum import Enum
from pydantic import BaseModel, Field, model_validator
from annotated_types import MinLen, MaxLen
import helper as h
from helper_restaurant import simulate_restaurant

## Definiowanie przestrzeni akcji i narzędzi 

Zanim przejdziemy do logiki `NextStep`, musimy zdefiniować "klocki", z których nasz agent będzie budował swoje plany. I tutaj musimy zrobić ważny inżynierski przystanek.

Pamiętasz lekcję 4.2? Mieliśmy tam bardzo precyzyjną maszynę stanów: `Greeting`, `Asking`, `Negotiating`, `Confirming`. Agent przeskakiwał z jednego konkretnego stanu do drugiego. To było świetne, żeby zrozumieć przepływ rozmowy, ale generowało sporo kodu.

Teraz, gdy wchodzimy głębiej w temat adaptacyjnego planowania i narzędzi, chcę ten kod uprościć. Gdybyśmy do tamtej złożonej struktury dołożyli jeszcze obsługę baz danych, utonęlibyśmy w linijkach kodu, tracąc z oczu to, co najważniejsze.

Dlatego w tej lekcji stosujemy świadome uproszczenie:

1.  Redukcja akcji konwersacyjnych: Zamiast prawie dziesięciu różnych stanów rozmowy, zostawiamy tylko trzy:
    - `Say` – worek na wszystko, co agent mówi (czy to powitanie, czy pytanie o stolik).
    - `Complete` – sukces
    - `Abandon` – porażka
    To pozwoli nam skupić się na *logice sterowania*, a nie na niuansach językowych (które zawsze później można tuningować).

2.  Wprowadzenie narzędzi (*Discovery Tools*): Zanim agent otworzy usta (`Say`), będzie miał możliwość "zajrzenia" do pamięci. Definiujemy dwa narzędzia:
    *   Jedno do pobrania danych z **pamięci semantycznej** (dane o kliencie, alergie, preferencje)
    *   Drugie do pobrania **pamięci epizodycznej** (historia zdarzeń z daną restauracją)

Dzięki temu zabiegowi, ciężar decyzyjny przesuwa się w ciekawe miejsce. Agent w `NextStep` nie będzie się już zastanawiał *"Czy mam się przywitać, czy zapytać?"*, ale stanie przed ważniejszym dylematem: "Czy mam już wystarczająco dużo informacji (Context), żeby w ogóle zacząć rozmowę, czy najpierw muszę użyć narzędzia?".

To jest esencja tej lekcji: pokazanie, jak planowanie przeplata się z działaniem i zdobywaniem wiedzy. Zobaczmy definicje tych struktur w kodzie.

In [4]:
# ───────────────────────────────────────────────────────────────────────────
# Discovery Tools 
# ───────────────────────────────────────────────────────────────────────────

class QueryCustomerData(BaseModel):
    """Narzędzie: Zapytaj system o dane klienta"""
    action: Literal["query_customer"]
    customer_id: str


class CheckHistory(BaseModel):
    """Narzędzie: Sprawdź historię z restauracją"""
    action: Literal["check_history"]
    customer_id: str
    restaurant: str


# ───────────────────────────────────────────────────────────────────────────
# Conversation Actions 
# ───────────────────────────────────────────────────────────────────────────

class Say(BaseModel):
    """Akcja: Powiedz coś do restauracji"""
    action: Literal["say"]
    utterance: str


class Complete(BaseModel):
    """TERMINAL: Zakończ z sukcesem"""
    action: Literal["complete"]
    summary: List[str]
    final_reservation: Dict[str, Any] = Field(default_factory=dict)


class Abandon(BaseModel):
    """TERMINAL: Zakończ porażka"""
    action: Literal["abandon"]
    reason: str

### ☝️ Analiza Inżynierska: Magia pola `Literal`

Zwróć uwagę na pole `action`, które pojawia się w każdej klasie (np. `action: Literal["say"]`). To nie jest przypadek.

W typowaniu Pythona i w strukturze Pydantic, użycie `Literal` pełni funkcję **dyskryminatora** -  czyli unikalnej etykiety, dzięki której system bezbłędnie rozpoznaje, z którym konkretnie narzędziem ma do czynienia.

Kiedy później wrzucimy te klasy do `NextStep` (używając `Union`), to właśnie to pole pozwoli naszemu *Routerowi* bezbłędnie rozpoznać, o którą klasę chodzi. Model AI nie musi "zgadywać" typu obiektu – on jawnie wybiera tag `say` albo `query_customer`.

Dzięki temu:
1.  Eliminujemy niejednoznaczność (marzenie w świecie LLM)
2.  Ułatwiamy parsing. Kod wie, że jeśli `action == "say"`, to należy szukać pola `utterance`.
3.  Wzmacniamy sterowalność. Model widzi te literale w definicji schematu JSON i traktuje je jak sztywne opcje wyboru (enum).

Warto też zauważyć, że `Say` ma pole `utterance`. To tutaj model wpisze tekst, który faktycznie zostanie wypowiedziany ("Dzień dobry, chciałbym zarezerwować..."). Narzędzia discovery (np. `QueryCustomerData`) nie mają pola `utterance`, bo działają "po cichu" – to myśli agenta, których restauracja nie słyszy (i to uruchamia się jeszcze przed tym jak zadzwoni).

## `NextStep` – Silnik decyzyjny

Mamy już zdefiniowane klocki (narzędzia i akcje). Teraz potrzebujemy architekta, który zadecyduje, jak ich użyć.

Tu wchodzi `NextStep`.

Na pierwszy rzut oka ta klasa wygląda banalnie. Trzy pola. Ale nie daj się zwieść tej prostocie. Im dłużej pracujesz z systemami agentowymi, tym bardziej doceniasz ten konkretny układ. To minimalistyczna, ale kompletna implementacja pętli decyzyjnej, która wymusza na modelu **myślenie przed działaniem**.


### Kluczowa mechanika
- Agent tworzy plan 0-5 kroków
- Wykonuje tylko pierwszy krok (to jest kluczowe, tylko jeden krok do przodu)
- Po feedback tworzy nowy plan (może być zupełnie inny!)
    
Spójrzmy na kod, a potem rozbierzemy go na czynniki pierwsze:

In [5]:
class NextStep(BaseModel):

    # KASKADA: Myślenie
    current_state: str = Field(
        description="Co się wydarzyło do tej pory? Co powiedziała restauracja?"
    )
    
    # KASKADA: Planowanie (0-5 kroków)
    plan_remaining_steps: Annotated[List[str], MinLen(0), MaxLen(5)] =  Field(
        description="Plan 0-5 kroków. Pusty = kończę. WYKONAM TYLKO PIERWSZY!"
    )

    # CYKL: Czy zmienił się plan od ostatniej tury?
    plan_changed: bool = Field(
        description="Czy to NOWY plan vs poprzedni? true = zmiana strategii"
    )
        
        
    # ROUTING: Wybór akcji do wykonania
    next_action: Annotated[
        
        Union[QueryCustomerData, CheckHistory, Say, Complete, Abandon],
        
        Field(
            discriminator='action',
            description="""
                Wykonaj PIERWSZY krok z planu.
                
                KRYTYCZNE - WORKFLOW:
                1. NIE WOLNO używać Say PRZED zakończeniem discovery!
                2. Wywołanie narzędzia które już było = POWAŻNY BŁĄD!
                
            """
        )
    ]

### Anatomia decyzji 

To, co widzisz powyżej, to skondensowane trzy wzorce projektowe w jednej strukturze. Zrozumienie ich współpracy jest kluczem do pojęcia "Agentic AI":

1.  **Kaskada (`current_state`)**:
    Model musi najpierw "uziemić się" w rzeczywistości. Zanim pozwoli mu się działać, zmuszamy go do analizy przeszłości. To zapobiega amnezji ("O co ja w ogóle pytałem?").

2.  **Cykl (`plan_remaining_steps`)**:
    Tutaj Cykl pełni inną rolę niż w prostym Researchu. Tutaj jest to **symulacja temporalna**. Używając listy `0-5`, zmuszamy model, by "uruchomił pętlę w wyobraźni" i przewidział sekwencję zdarzeń.
    *   Dlaczego to ważne? Bo model, który widzi 5 kroków do przodu, rzadziej popełnia błąd w kroku pierwszym.
    *   Pole `plan_changed` dodatkowo wymusza na nim refleksję: *"Czy trzymam się starego planu, czy właśnie muszę go wyrzucić do kosza, bo odkryłem, że klient ma alergię?"*.

3.  **Router (`next_action`)**:
    Na samym końcu lejka decyzyjnego jest twardy wybór. Użycie `Union` wraz z `discriminator='action'` działa jak zwrotnica kolejowa. Model nie może odpowiedzieć "trochę zapytam, a trochę sprawdzę". Musi wybrać **jedną** konkretną klasę Pydantic. To daje nam 100% sterowalności w kodzie.

```text

┌─────────────────────────────────────────────────────────────────────────────┐
│                              NEXT STEP (LLM Output)                         │
│                                                                             │
│  ┌─────────────────────────────────────────────────────────────────────┐    │
│  │ 🧠 KASKADA: Myślenie                                                │    │
│  │    current_state: "Co się wydarzyło? Co powiedziała restauracja?"   │    │
│  └─────────────────────────────────────────────────────────────────────┘    │
│                                      │                                      │
│                                      ▼                                      │
│  ┌─────────────────────────────────────────────────────────────────────┐    │
│  │ 📋 KASKADA: Planowanie                                              │    │
│  │    plan_remaining_steps: List[str]  (0-5 kroków)                    │    │
│  │    ⚠️  WYKONAM TYLKO PIERWSZY KROK!                                 │    │
│  └─────────────────────────────────────────────────────────────────────┘    │
│                                      │                                      │
│                                      ▼                                      │
│  ┌─────────────────────────────────────────────────────────────────────┐    │
│  │ 🔄 CYKL: Tracking                                                   │    │
│  │    plan_changed: bool  (true = zmiana strategii)                    │    │
│  └─────────────────────────────────────────────────────────────────────┘    │
│                                      │                                      │
│                                      ▼                                      │
│  ┌─────────────────────────────────────────────────────────────────────┐    │
│  │ 🎯 ROUTING: Wybór akcji                                             │    │
│  │    next_action: Union[                                              │    │
│  │        QueryCustomerData,                                           │    │
│  │        CheckHistory,                                                │    │
│  │        Say,                                                         │    │
│  │        Complete,                                                    │    │
│  │        Abandon                                                      │    │
│  │    ]                                                                │    │
│  └─────────────────────────────────────────────────────────────────────┘    │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
                                       
                                       
```                                       



W ten sposób `NextStep` staje się mostem między "myśleniem" (Kaskada + Cykl planowania) a "działaniem" (Router wykonawczy).

## Most między decyzją a działaniem: `dispatch()`

Skoro mamy już zdefiniowany "centrum decyzyjne" (`NextStep`), potrzebujemy rąk (lub czasem mówi się "kleju").


Zrozum to dobrze: LLM, korzystając z mechanizmu **Structured Output**, wypełnia nam obiekt `NextStep`. To jest ten moment autonomii – model decyduje, którą ścieżkę wybrać. Ale sama *decyzja* to jeszcze nie *działanie*. Obiekt Pydantic w pamięci RAM nikomu stolika nie zarezerwuje.

Musimy przekazać tę decyzję do warstwy wykonawczej, którą nazwiemy **`dispatch`**.

### Pętla główna
```text
    ┌───────────────────────────────────────────────────────────────┐
    │                                                               │
    │   ┌─────────┐    ┌───────────┐    ┌───────────┐    ┌──────┐   │
    │   │   LLM   │───▶│ NextStep  │───▶│ dispatch()│───▶│result│   │
    │   └─────────┘    └───────────┘    └───────────┘    └──────┘   │
    │        ▲                                              │       │
    │        │                                              │       │
    │        └──────────────────────────────────────────────┘       │
    │                    (jeśli NIE Terminal)                       │
    │                                                               │
    └───────────────────────────────────────────────────────────────┘
    
    STOP gdy: Complete ✅ lub Abandon ❌
    
```    



To jest powrót do świata **Software 1.0**. Tutaj kończy się prawdopodobieństwo, a zaczyna twarda inżynieria. Niezależnie od tego, czy wywołujesz zewnętrzne API (z lub bez MCP), czy robisz prosty lookup w słowniku – musisz tę operację ustabilizować.

Funkcja `dispatch` pełni rolę bezpiecznika i tłumacza. Przyjmuje surową decyzję modelu (np. `QueryCustomerData`) i zamienia ją na konkretny wynik (string), który wróci do modelu jako Feedback.

Spójrz na implementację (na razie uproszczoną, z "mockowanymi" danymi):

In [4]:
def dispatch(action: BaseModel, discoveries: List[str]) -> str:
    # ───────────────────────────────────────────────────────────────────────
    # DISCOVERY TOOLS - odkrywają wiedzę
    # ───────────────────────────────────────────────────────────────────────

    if isinstance(action, QueryCustomerData):
        # Szukamy po `customer_id`
        # Odkryj alergię → dodaj do discoveries!
        
        customer = {"name": "Jan", "phone": 123}
        return f"✅ Dane klienta: {customer['name']}, brak alergii, tel: {customer['phone']}"

    if isinstance(action, CheckHistory):
        # Szukamy w historii restauracje
        
        return f"✅ Brak problemów w historii"
        

    # ───────────────────────────────────────────────────────────────────────
    # CONVERSATION - rozmowa z restauracją
    # ───────────────────────────────────────────────────────────────────────

    if isinstance(action, Say):
        # simulate_restaurant
        return "Restaurcja coś odpowada!"


    # ───────────────────────────────────────────────────────────────────────
    # TERMINAL - zakończenie
    # ───────────────────────────────────────────────────────────────────────

    if isinstance(action, Complete):
        return "✅ SUCCESS"

    if isinstance(action, Abandon):
        return "❌ FAILED"

    return "⚠️  Unknown action"

### 🧠 Co tu się dzieje?
Zwróć uwagę na strukturę `if isinstance(...)`. To jest bezpośrednia obsługa naszego **Routera**. Ponieważ w `NextStep` użyliśmy `Union`, tutaj po prostu "rozpakowujemy" decyzję modelu i przypisujemy do niej konkretną logikę Pythonową. To jest ten moment, gdzie LLM to (intencja) spotyka się z Software 1.0 (implementacja).


```text

                    ┌──────────────────────────────────┐
                    │         DISPATCH(action)         │
                    │      + discoveries: List[str]    │
                    └──────────────────────────────────┘
                                       │
          ┌────────────────────────────┼────────────────────────────┐
          │                            │                            │
          ▼                            ▼                            ▼
┌─────────────────────┐   ┌─────────────────────┐   ┌─────────────────────┐
│  🔍 DISCOVERY TOOLS │   │  💬 CONVERSATION    │   │  🏁 TERMINAL        │
│  (odkrywają wiedzę) │   │  (rozmowa)          │   │  (zakończenie)      │
├─────────────────────┤   ├─────────────────────┤   ├─────────────────────┤
│                     │   │                     │   │                     │
│ QueryCustomerData   │   │ Say                 │   │ Complete            │
│    │                │   │    │                │   │    │                │
│    ▼                │   │    ▼                │   │    ▼                │
│ ┌─────────────────┐ │   │ ┌─────────────────┐ │   │ ┌─────────────────┐ │
│ │ customer_id     │ │   │ │ simulate_       │ │   │ │ "✅ SUCCESS"    │ │
│ │      ↓          │ │   │ │ restaurant()    │ │   │ └─────────────────┘ │
│ │ Dane + alergie  │ │   │ │      ↓          │ │   │                     │
│ │ → discoveries!  │ │   │ │ "Restauracja    │ │   │ Abandon             │
│ └─────────────────┘ │   │ │  odpowiada!"    │ │   │    │                │
│                     │   │ └─────────────────┘ │   │    ▼                │
│ CheckHistory        │   │                     │   │ ┌─────────────────┐ │
│    │                │   │                     │   │ │ "❌ FAILED"     │ │
│    ▼                │   │                     │   │ └─────────────────┘ │
│ ┌─────────────────┐ │   │                     │   │                     │
│ │ Historia        │ │   │                     │   │                     │
│ │ restauracji     │ │   │                     │   │                     │
│ │      ↓          │ │   │                     │   │                     │
│ │ "Brak problemów"│ │   │                     │   │                     │
│ └─────────────────┘ │   │                     │   │                     │
└─────────────────────┘   └─────────────────────┘   └─────────────────────┘
          │                            │                            │
          └────────────────────────────┼────────────────────────────┘
                                       │
                                       ▼
                          ┌────────────────────────┐
                          │    RETURN: string      │
                          │    (wynik do LLM)      │
                          └────────────────────────┘
                          
```                          

## Dlaczego kodujemy `dispatch` ręcznie?

Patrząc na powyższy kod, możesz poczuć pewien niedosyt. "Zaraz, przecież są odważne pomysły (typu *Code Mode*) aby generować kod w locie! Czy nie byłoby bardziej elastycznie, gdyby model sam sobie napisał funkcję `dispatch` i wykonał ją `exec()`?"

Pokusa jest silna! Wyobraźnia podpowiada systemy, które same piszą swoje sterowniki w czasie rzeczywistym.

Jednak na poziomie 5 (Strategic AI) musimy wylać na głowę kubeł zimnej wody. Świadomie rezygnujemy z generowania kodu w tej warstwie.

Dlaczego?

Ponieważ `dispatch` to Twoja ostatnia linia obrony. W architekturze agentowej model (LLM) odpowiada za intencję (czyli "co chcę zrobić"), ale to Ty, inżynier, odpowiadasz za wykonanie (czyli "jak to zrobić bezpiecznie").

1.  **Kontrola i walidacja:** `dispatch` to miejsce, gdzie umieszczasz twarde reguły biznesowe. To tu sprawdzasz, czy ID klienta nie jest SQL Injection. To tu blokujesz próbę rezerwacji stolika na rok 1900. Gdyby ten kod generował AI, straciłbyś kontrolę nad tymi bezpiecznikami.
2.  **Determinizm:** Mózg agenta (`NextStep`) jest probabilistyczny – może zmienić zdanie, wymyślić nową strategię. To wystarczająco dużo zmienności. Ciało agenta (`dispatch`) musi być deterministyczne. Jeśli agent naciska "hamulec", samochód musi hamować, a nie zastanawiać się, czy może lepiej skręcić.
3.  **Jeszcze tam nie jesteśmy:** Brutalna prawda jest taka, że nie jesteśmy jeszcze na tyle odważni (ani technologie nie są na tyle stabilne), aby pozwolić systemom produkcyjnym na nadpisywanie własnego kodu wykonawczego w pętli.

Dlatego na poziomie 5 obowiązuje zasada: elastyczne planowanie, sztywne wykonanie. `dispatch` pozostaje w domenie Software 1.0 – jako solidny fundament, na którym tańczy sztuczna inteligencja.

## Pamięć (uproszczona wersja)

Zanim zaimplementujemy logikę w `dispatch`, musimy mieć **na czym** pracować. Nasz agent ma narzędzia do "odkrywania wiedzy" (`QueryCustomerData`, `CheckHistory`), ale te narzędzia muszą mieć skąd tę wiedzę czerpać.

W środowisku produkcyjnym w tym miejscu podłączylibyśmy się do X (gdzie X może być dowolny system bazodanowy lub jeszcze cokolwiek innego) oraz do bazy logów Y (np. Elasticsearch/Postgres). Jednak w środowisku deweloperskim i edukacyjnym stosujemy technikę *mockowania*.

Stworzymy teraz proste, statyczne słowniki, które będą udawać nasze bazy danych. Podzielimy je zgodnie z teorią pamięci, o której mówiliśmy w Module 2:
1.  **Pamięć semantyczna:** Fakty o kliencie (trwałe, rzadko zmienne).
2.  **Pamięć epizodyczna:** Historia zdarzeń (co się wydarzyło w przeszłości).


### `🧠 SEMANTIC MEMORY` - "Co wiem o świecie i użytkowniku" (trwała wiedza)

In [6]:
class CustomerProfile(BaseModel):
    """Profil klienta - trwała wiedza w systemie CRM"""
    name: str
    phone: str = ""  # Numer telefonu klienta (niezbędny do rezerwacji)
    allergies: List[str] = []
    preferences: List[str] = []

# Mock bazy danych klientów (klucz = customer_id)
CUSTOMERS_DB = {
    "kowalski": CustomerProfile(
        name="Jan Kowalski",
        phone="+48 123 456 789",
        allergies=["orzechy"], # <--- Krytyczna informacja!
        preferences=["stolik przy oknie"]
    ),
    "nowak": CustomerProfile(
        name="Anna Nowak",
        phone="+48 987 654 321",
        allergies=["gluten"],
        preferences=[]
    )
}

### `📚 EPISODIC MEMORY` - "Co się wydarzyło w przeszłości" (historia)

In [7]:
# Mock historii zdarzeń (klucz = (customer_id, restaurant_name))
RESTAURANT_HISTORY = {
    ("kowalski", "Pod Różą"): {
        "date": "2024-02-20",
        "problem": "Klient czekał 30 min mimo rezerwacji", # <--- Agent powinien to wiedzieć!
        "severity": "high"
    },
    ("kowalski", "Bella Italia"): {
        "date": "2024-01-15",
        "problem": "Nie przygotowano stolika mimo rezerwacji",
        "severity": "high"
    }
}

Dodatkowo zdefiniujemy `ReservationGoal` – czyli strukturę, która reprezentuje "życzenie" klienta (Input), uruchamiające całą lawinę zdarzeń.

In [8]:
class ReservationGoal(BaseModel):
    """Cel rezerwacji"""
    restaurant: str
    date: str
    time: str
    party_size: int
    flexibility_min: int = 30

### Analiza danych

Co my tu właściwie mamy?

1.  **Pułapka w pamięci semantycznej (`CUSTOMERS_DB`):** Zauważ, że Jan Kowalski ma **alergię na orzechy**. Jeśli nasz agent zadzwoni do restauracji i tego nie zgłosi, to z inżynierskiego punktu widzenia system nie działa poprawnie (*Safety Issue*). Dlatego narzędzie `QueryCustomerData` jest niezbędne przed wykonaniem telefonu.

2.  **Ostrzeżenie w pamięci epizodycznej (`RESTAURANT_HISTORY`):** Widzimy, że Jan Kowalski miał już problem z restauracją "Pod Różą". To zmienia kontekst negocjacji. Agent, mając tę wiedzę, powinien być bardziej asertywny i upewnić się, że tym razem stolik na pewno będzie czekał.

3.  **Cel (`ReservationGoal`):** To jest nasz punkt startowy. To czyste dane wejściowe ("Chcę stolik na 5 osób"). Agent weźmie ten obiekt i zacznie go "obudowywać" wiedzą zdobytą z dwóch powyższych baz, zanim podejmie jakiekolwiek działanie na zewnątrz.

## Zarządzanie kontekstem: Mechanizm `discoveries`

Zanim przejdziemy do zadania, musimy omówić jeden prosty, ale przydatny mechanizm zarządzania stanem.

Zauważ, że nasza funkcja `dispatch` przyjmuje argument `discoveries: List[str]`.
To nasz "plecak", który przekazujemy agentowi. Na początku jest pusty `[]`, ale w trakcie działania agenta będziemy do niego wrzucać (appendować) znalezione fakty.

Dlaczego tak? Bo agent jest bezstanowy. Jeśli w kroku 1 użyje narzędzia i dowie się, że "Klient ma alergię", a my tego nie zapiszemy i nie przekażemy do kolejnego promptu, to w kroku 2 agent o tym zapomni.

W tym celu stosujemy prostą technikę. Wewnątrz funkcji `dispatch`, gdy narzędzie coś zwróci, dokonujemy ekstrakcji kluczowych faktów i doklejamy je do listy.

- Jeśli pobierzesz profil klienta -> wyciągnij alergię i telefon -> zrób `discoveries.append("ODKRYTO: Alergia na orzechy")`.
- Jeśli sprawdzisz historię -> znajdziesz problem -> zrób `discoveries.append("ODKRYTO: Problem w przeszłości...")`.

Te "doklejone" stringi posłużą nam później do dynamicznego zbudowania Promptu (System Context).

### 💡 Dygresja inżynierska (Context Window Trade-offs)
Swoją drogą, możesz się zastanawiać: *"Dlaczego po prostu nie wrzucimy całego obiektu profilu klienta do promptu na starcie? Po co bawić się w odkrywanie?"*

To bardzo celne pytanie. W wielu prostych systemach produkcyjnych **dokładnie tak robimy**. Jeśli profil klienta jest mały, wrzucamy go w całości do `System Prompt` ("Tu są dane klienta: JSON..."). To upraszcza architekturę i eliminuje jeden krok (narzędzie).

Jednak tutaj uczymy się czegoś ważniejszego: **selektywności**.
Wyobraź sobie, że profil klienta to nie 5 pól, ale 500 stron dokumentacji medycznej albo historia 10 lat transakcji bankowych. Wtedy nie możesz "wrzucić wszystkiego". Musisz nauczyć agenta mechanizmu:
1.  Pomyśl, czego potrzebujesz
2.  Sięgnij po konkretną daną (narzędzie)
3.  Wyciągnij tylko to, co ważne (ekstrakcja do `discoveries`)

Dlatego w tym ćwiczeniu, mimo że baza jest mała, symulujemy pracę z dużym systemem. Uczymy agenta aktywnie szukać faktów, a nie tylko je przetwarzać.

## 🧠 Zadanie 4.3.1: Podłączenie Pamięci Semantycznej

Zaczynamy od bezpieczeństwa. Twój Agent nie może zadzwonić do restauracji, nie wiedząc, czy klient nie wyląduje w szpitalu po zjedzeniu deseru. Musisz obsłużyć narzędzie `QueryCustomerData`.

**Twoim zadaniem jest:**
1.  Wewnątrz funkcji `dispatch` dodać blok sprawdzający, czy akcja to `QueryCustomerData`.
2.  Pobrać profil z `CUSTOMERS_DB` używając `customer_id`.
3.  **Krytyczne:** Jeśli profil zawiera alergie, musisz sformatować je do stringa i dodać do listy `discoveries` (np. `"ODKRYTO: Alergia na X"`).
4.  Dodać do `discoveries` numer telefonu (jest niezbędny do rezerwacji).
5.  Zwrócić string zwrotny dla modelu (np. `"✅ Pobrano dane: Jan Kowalski"`).

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj, aby zobaczyć podpowiedź
    </summary>
<p>

Pamiętaj, że `discoveries` to zwykła lista Pythonowa. Użyj `.append()`.
Aby ładnie wyświetlić alergie, możesz użyć: `", ".join(profile.allergies)`.

```python
    if isinstance(action, QueryCustomerData):
        # 1. Pobierz z bazy
        profile = CUSTOMERS_DB.get(action.customer_id)
        
        # 2. Obsłuż błąd (brak klienta)
        if not profile:
            return "❌ BŁĄD: Nie znaleziono klienta."

        # 3. Zaktualizuj discoveries (Side Effect)
        # ... tu Twój kod ...

        return f"✅ Pobrano dane: {profile.name}"
```
</p>
<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj, aby zobaczyć rozwiązanie
    </summary>
<p>

```python
    if isinstance(action, QueryCustomerData):
        profile = CUSTOMERS_DB.get(action.customer_id)
        if not profile:
            return f"❌ Error: Customer {action.customer_id} not found"

        # Telefon jest zawsze potrzebny
        discoveries.append(f"ODKRYTO: Telefon klienta: {profile.phone}")
        
        # Alergie są krytyczne - logujemy je wyraźnie
        if profile.allergies:
            allergy_str = ", ".join(profile.allergies)
            discoveries.append(f"ODKRYTO: ⚠️ Klient ma ALERGIĘ na: {allergy_str}")

        return f"✅ Pobrano dane klienta: {profile.name}"
```
</p>
</details>
</details>

In [ ]:
## YOUR CODE HERE

## 🧠 Zadanie 4.3.2: Podłączenie pamięci epizodycznej

Teraz dodajmy wymiar czasu. Agent musi wiedzieć, czy dana restauracja nie sprawiała problemów w przeszłości. Obsłuż narzędzie `CheckHistory`.

**Twoim zadaniem jest:**
1.  Wewnątrz `dispatch` dodać blok dla `CheckHistory`.
2.  Skonstruować klucz do słownika `RESTAURANT_HISTORY`. Uwaga: kluczem jest krotka (tuple): `(id_klienta, nazwa_restauracji)`.
3.  Sprawdzić, czy istnieje wpis pod tym kluczem.
4.  Jeśli tak – to ważny sygnał ostrzegawczy! Dodaj wpis do `discoveries` z opisem problemu (np. `"ODKRYTO: Ostatnio czekaliśmy 30 min"`).
5.  Zwróć odpowiedni komunikat zwrotny (inny dla czystej historii, inny dla wykrytego problemu).

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj, aby zobaczyć podpowiedź
    </summary>
<p>

Największą pułapką jest tutaj klucz słownika. Musisz go zbudować ręcznie z danych dostępnych w obiekcie akcji.

```python
    if isinstance(action, CheckHistory):
        # Budujemy klucz (tuple)
        key = (action.customer_id, action.restaurant)
        
        # Sprawdzamy w bazie
        record = RESTAURANT_HISTORY.get(key)
        
        # ... logika if record: ...
```
</p>
<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj, aby zobaczyć rozwiązanie
    </summary>
<p>

```python
    if isinstance(action, CheckHistory):
        # Kluczem w naszej "bazie" jest krotka (user, restaurant)
        key = (action.customer_id, action.restaurant)
        record = RESTAURANT_HISTORY.get(key)
        
        if record:
            # Znaleziono incydent - musimy o tym pamiętać!
            # Dodajemy do discoveries, żeby Agent widział to przy generowaniu rozmowy
            discoveries.append(f"ODKRYTO: ⚠️ Historia problemów w {action.restaurant}: {record['problem']}")
            return f"⚠️ Znaleziono incydent z dnia {record['date']}: {record['problem']}"
        
        return f"✅ Historia czysta. Brak odnotowanych problemów z {action.restaurant}."
```
</p>
</details>
</details>

In [ ]:
## YOUR CODE HERE

## Scenariusze symulacji restauracji.

W prawdziwym świecie, gdy dzwonisz do restauracji, po drugiej stronie słuchawki jest człowiek, którego reakcji nie możesz przewidzieć. Aby jednak móc zbudować i przetestować nasz system, musimy to środowisko zasymulować.

W tym celu wprowadzamy koncepcję bliźniaczego systemu (*Shadow System*). Będziemy uruchamiać de facto dwa modele równolegle:
1.  **Główny agent (Twój kod):** To nad nim pracujemy
2.  **Symulator restauracji (Helper):** To prosty skrypt oparty na LLM, który "udaje" recepcję. Jego implementacja jest ukryta w pliku `helper_restaurant.py`, bo nie jest przedmiotem tej lekcji

Dla nas ważne jest tylko jedno: chcemy mieć kontrolę nad tym symulatorem. Nie chcemy, żeby restauracja losowo odrzucała rezerwację, gdy testujemy *Happy Path*.

Dlatego definiujemy `Enum`, który działa jak "przełącznik rzeczywistości". Przekazując go do funkcji `dispatch` (a potem do helpera), zmuszamy symulator do odegrania konkretnej roli.

In [ ]:
class RestaurantScenario(str, Enum):
    """
    Scenariusze symulacji restauracji.
    """
    
    # 🟢 Sytuacja idealna
    HAPPY_PATH = "happy_path"              # Wszystko OK, rezerwacja potwierdzona od razu.
    
    # 🔴 Sytuacje brzegowe (testowanie porażki/Abandon)
    FULLY_BOOKED = "fully_booked"          # Zajęte, brak alternatywy. Agent powinien się poddać.
    MIN_PARTY_SIZE = "min_party_size"      # Np. "Przyjmujemy tylko grupy od 4 osób".
    ALLERGY_PROBLEM = "allergy_problem"    # "Nie możemy zagwarantować braku orzechów". Safety Check.
    
    # 🟡 Sytuacje negocjacyjne (testowanie adaptacji/NextStep)
    ALTERNATIVE_TIME = "alternative_time"  # "19:00 zajęte, ale mamy 19:30". Agent musi sprawdzić flexibility.
    REQUIRE_PREPAYMENT = "require_prepayment" # "OK, ale wymagamy 50 zł zadatku". Agent musi zapytać użytkownika lub podjąć decyzję.

### Po co nam to?
Dzięki temu mechanizmowi, uruchamiając Agenta, możemy powiedzieć: *"Sprawdźmy, jak sobie poradzisz, gdy nie ma miejsc"*. Ustawiamy flagę `FULLY_BOOKED` i obserwujemy, czy Agent wpada w pętlę, czy elegancko kończy rozmowę akcją `Abandon`. To pozwala nam testować logikę adaptacji w izolowanych warunkach.


### Co my tu testujemy?

Każdy z tych scenariuszy celuje w inny fragment logiki `NextStep`:

1.  **`HAPPY_PATH`**: Czy agent potrafi przejść prostą ścieżkę: Wykryj -> Zadzwoń -> Potwierdź -> Zakończ.
2.  **`FULLY_BOOKED`**: Czy agent potrafi zrozumieć odmowę i użyć akcji `Abandon` zamiast bez sensu pytać dalej.
3.  **`ALTERNATIVE_TIME`**: To jest **najważniejszy test inteligencji**. Agent musi:
    *   Zrozumieć nową propozycję (np. 19:30).
    *   Porównać ją ze swoim `ReservationGoal` (gdzie `flexibility_min` = 30).
    *   Podjąć decyzję: "To się mieści w granicach, biorę" LUB "To za późno, odmawiam".

Dzięki temu Enumowi, nasze testy stają się wygodniejsze. 

## Kompletna funkcja `dispatch`

Skoro masz już gotowe klocki z zadań (obsługa `QueryCustomerData` i `CheckHistory`), czas złożyć kompletną funkcję `dispatch`.

Zwróć uwagę na jedną istotną zmianę w sygnaturze funkcji. Przyjmujemy teraz aż cztery argumenty:
`action`, `discoveries`, `goal` oraz `scenario`.

**Dlaczego?**
W świecie rzeczywistym, gdy dzwonisz do restauracji, ona "po prostu jest". W naszym środowisku testowym musimy ją symulować.
- Parametry `goal` i `scenario` są przekazywane głównie na potrzeby funkcji `simulate_restaurant`.
- Symulator musi wiedzieć, o co pyta klient (`goal`), żeby wiedzieć, czy odmówić (bo np. `scenario` to "brak miejsc"), czy potwierdzić.

Gdybyśmy łączyli się z prawdziwym API (np. Vapi czy Twilio), te parametry mogłyby zniknąć z `dispatch`. Ale tutaj trzymamy się zasady: **Testowalność przede wszystkim.**

Tu masz finalny kod, który łączy Twoją pracę z obsługą rozmowy:

In [ ]:
def dispatch(
    action: BaseModel, 
    discoveries: List[str],
    goal: ReservationGoal,  # ← DODAJ
    scenario: RestaurantScenario  # ← DODAJ
) -> str:


    # ───────────────────────────────────────────────────────────────────────
    # DISCOVERY TOOLS - odkrywają wiedzę
    # ───────────────────────────────────────────────────────────────────────

    if isinstance(action, QueryCustomerData):
        profile = CUSTOMERS_DB.get(action.customer_id)
        if not profile:
            return f"❌ Error: Customer {action.customer_id} not found"

        # Telefon jest zawsze potrzebny
        discoveries.append(f"ODKRYTO: Telefon klienta: {profile.phone}")
        
        # Alergie są krytyczne - logujemy je wyraźnie
        if profile.allergies:
            allergy_str = ", ".join(profile.allergies)
            discoveries.append(f"ODKRYTO: ⚠️ Klient ma ALERGIĘ na: {allergy_str}")

        return f"✅ Pobrano dane klienta: {profile.name}"
    

    if isinstance(action, CheckHistory):
        # Kluczem w naszej "bazie" jest krotka (user, restaurant)
        key = (action.customer_id, action.restaurant)
        record = RESTAURANT_HISTORY.get(key)
        
        if record:
            # Znaleziono incydent - musimy o tym pamiętać!
            # Dodajemy do discoveries, żeby Agent widział to przy generowaniu rozmowy
            discoveries.append(f"ODKRYTO: ⚠️ Historia problemów w {action.restaurant}: {record['problem']}")
            return f"⚠️ Znaleziono incydent z dnia {record['date']}: {record['problem']}"
        
        return f"✅ Historia czysta. Brak odnotowanych problemów z {action.restaurant}."

    # ───────────────────────────────────────────────────────────────────────
    # CONVERSATION - rozmowa z restauracją
    # ───────────────────────────────────────────────────────────────────────

    if isinstance(action, Say):
        return simulate_restaurant(action.utterance, scenario, goal)

    # ───────────────────────────────────────────────────────────────────────
    # TERMINAL - zakończenie
    # ───────────────────────────────────────────────────────────────────────

    if isinstance(action, Complete):
        return "✅ SUCCESS"

    if isinstance(action, Abandon):
        return "❌ FAILED"

    return "⚠️  Unknown action"

## Co tu się dzieje w sekcji `Say`?

```python
if isinstance(action, Say):
        return simulate_restaurant(action.utterance, scenario, goal)
```    

Zamiast pisać skomplikowane if-y symulujące rozmowę wewnątrz `dispatch`, delegujemy to zadanie do `simulate_restaurant`. To dobra praktyka – `dispatch` ma być routerem, a nie miejscem, gdzie trzymasz logikę całego świata. Dla nas najważniejsze jest to, że `Say` zwraca stringa (odpowiedź recepcjonisty), który w kolejnym kroku trafi do mózgu Agenta.

## Dynamiczny system prompt

Mamy już silnik decyzyjny (`NextStep`) i ramię wykonawcze (`dispatch`). Czego brakuje? Instrukcji operacyjnej.

W typowych tutorialach o AI `System Prompt` to stała zmienna na początku pliku: `const SYSTEM_PROMPT = "Jesteś pomocnym asystentem..."`. W **Software 3.0** i systemach agentowych to nie wystarczy.

Nasz Agent zmienia się w czasie:
- W `T=0` nic nie wie
- W `T=1` wie już, że klient ma alergię (bo zadziałał `dispatch`)
- W `T=2` wie, że restauracja odmówiła (bo zadziałał symulator)

Dlatego nasz *prompt* nie może być stałą. Musi być funkcją, która jest wywoływana na nowo przed każdą decyzją Agenta. Funkcja ta bierze aktualny stan świata (`discoveries`, `last_response`) i "piecze" z niego świeżą instrukcję dla modelu.

## 🧠 Zadanie 4.3.3: Implementacja funkcji `build_prompt`

Twoim zadaniem jest stworzenie funkcji, która dynamicznie składa prompt.

### Wymagania:
1.  Funkcja przyjmuje listę `discoveries` i string `last_response`.
2.  Jeśli są jakieś odkrycia, sformatuj je i wstrzyknij do promptu. To kluczowe – bez tego Agent "zapomni", co odkrył przed chwilą.
3.  Jeśli jest odpowiedź z restauracji, doklej ją na końcu.
4.  W treści promptu zdefiniuj sztywny **Workflow**: Najpierw Discovery, potem Conversation.

> **💡 Note Inżynierski:** W dużych systemach produkcyjnych do tego celu używamy silników szablonów, takich jak **Jinja2** (oddzielając logikę od tekstu). Tutaj, aby nie skakać między plikami i zachować prostotę edukacyjną, użyjemy zwykłego Pythonowego f-stringa. Działa tak samo, a widzisz wszystko w jednym miejscu.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj, aby zobaczyć rozwiązanie
    </summary>
<p>

```python
def build_prompt(discoveries: List[str], last_response: str = "") -> str:
    """
    Buduje prompt z odkryć - PROSTY f-string!

    KLUCZOWE: Prompt aktualizowany CO TURĘ z nowymi odkryciami.
    To jest mechanizm adaptacji - LLM widzi NOWE fakty i tworzy NOWY plan.
    """

    # 1. Formatowanie pamięci krótkotrwałej (Discoveries)
    discoveries_text = ""
    if discoveries:
        # Bierzemy ostatnie 5 faktów, żeby nie przeładować kontekstu (prosta heurystyka)
        facts = "\n".join(f"  • {d}" for d in discoveries)
        discoveries_text = f"\n\nODKRYTE FAKTY (wykorzystaj to!):\n<facts>{facts}</facts>"

    # 2. Formatowanie feedbacku ze świata (Observation)
    last_text = ""
    if last_response:
        last_text = f"\n\nOSTATNIA ODPOWIEDŹ RESTAURACJI:\n  → <last_response>{last_response}</last_response>"

    # 3. Złożenie całości
    return f"""Jesteś agentem rezerwacji stolika.

═══════════════════════════════════════════════════════════════════════
WORKFLOW DISCOVERY (MUSISZ TO ZROBIĆ NA POCZĄTKU!)
═══════════════════════════════════════════════════════════════════════

ETAP 1 - DISCOVERY (wywołaj narzędzia):
  1. QueryCustomerData → odkryj dane klienta (alergie, preferencje, telefon)
  2. CheckHistory → sprawdź historię z restauracją (problemy?)

ETAP 2 - CONVERSATION (dopiero teraz!):
  3. Say → rozmawiaj z restauracją
     - Wspomnij o alergii JEŚLI została odkryta
     - Zapytaj o przeszły problem JEŚLI był odkryty
     - Podaj nazwisko klienta
     - Potwierdź szczegóły (data, godzina, liczba osób)

ETAP 3 - COMPLETION:
  4. Gdy restauracja potwierdzi rezerwację → Complete

═══════════════════════════════════════════════════════════════════════
ADAPTACYJNE PLANOWANIE (ReAct Pattern)
═══════════════════════════════════════════════════════════════════════

• Stwórz plan 1-5 kroków (patrzysz w przyszłość)
• Wykonaj TYLKO pierwszy krok (działasz tu i teraz)
• Po feedback stwórz NOWY plan (może być zupełnie inny!)
• Plan NIE JEST ŚWIĘTOŚCIĄ - zmień go gdy odkryjesz nowe fakty!
{discoveries_text}{last_text}

═══════════════════════════════════════════════════════════════════════
CO TERAZ ZROBISZ?
═══════════════════════════════════════════════════════════════════════"""
```

</p>
</details>


In [ ]:
## YOUR CODE HERE

### Analiza Promptu
To jest Twoje centrum sterowania. Zauważ sekcję **WORKFLOW DISCOVERY**.
Agent (LLM) z natury jest leniwy i chce od razu przejść do rozmowy (`Say`). Musisz go "zmusić" promptem, żeby najpierw wykonał nudną pracę sprawdzania danych (`QueryCustomerData`).
Dopiero gdy `discoveries` się wypełnią (i pojawią się w prompcie w kolejnej iteracji), Agent "zauważy", że etap 1 jest zakończony i przejdzie do etapu 2.

## Pętla główna 

Mamy już komplet klocków:
1.  **Centrum:** `NextStep` (planowanie)
2.  **Ręce:** `dispatch` (wykonanie)
3.  **Pamięć:** `discoveries` (kontekst)
4.  **Instrukcja:** `build_prompt` (świadomość)

Czas to spiąć w całość. Poniższa funkcja `make_reservation` to serce naszego agenta. To pętla, która napędza proces ReAct.

W przeciwieństwie do prostych chatbotów, gdzie po prostu doklejamy wiadomości do listy, tutaj dzieje się coś ważniejszego: dynamiczna aktualizacja system promptu (czyli *Context Engineering*). W każdej turze agent "budzi się" z nową wiedzą o świecie, którą wstrzykujemy mu prosto do instrukcji systemowej.


```text
┌══════════════════════════════════════════════════════════════════════════════┐
│                        🚀 START: make_reservation()                          │
│                                                                              │
│   INPUT:  customer_id, goal (RestaurantGoal), scenario                       │
└══════════════════════════════════════════════════════════════════════════════┘
                                       │
                                       ▼
┌──────────────────────────────────────────────────────────────────────────────┐
│              1️⃣  INICJALIZACJA STANU SESJI                                   │
│ ─────────────────────────────────────────────────────────────────────────────│
│                                                                              │
│   discoveries = []          ← 🧠 Wiedza odkryta (pusta na start)             │
│                               📝 Będzie MUTOWANA przez dispatch!             │
│                                                                              │
│   messages = []             ← 💬 Historia rozmowy dla LLM                    │
│   last_response = ""        ← 🏪 Ostatnia odpowiedź restauracji              │
│                                                                              │
│   ┌─────────────────────────────────────────────────────────────────────┐    │
│   │  messages[0] = {role: "system", content: build_prompt([], "")}      │    │
│   │  messages[1] = {role: "user", content: "Zarezerwuj stolik..."}      │    │
│   └─────────────────────────────────────────────────────────────────────┘    │
│                                                                              │
└──────────────────────────────────────────────────────────────────────────────┘
                                       │
                                       ▼
          ╔════════════════════════════════════════════════════════════╗
          ║           2️⃣  GŁÓWNA PĘTLA  (for i in range(10))           ║
          ║                                                            ║
          ╠════════════════════════════════════════════════════════════╣
          ║                                                            ║
          ║    ┌────────────────────────────────────────────────┐      ║
          ║    │         TURA i (max 10 iteracji)               │      ║
          ║    └────────────────────────────────────────────────┘      ║
          ║                          │                                 ║
          ╚══════════════════════════╪═════════════════════════════════╝
                                     │
                                     ▼
┌──────────────────────────────────────────────────────────────────────────────┐
│                    KROK A: 📝 AKTUALIZACJA PROMPTU                           │
│ ─────────────────────────────────────────────────────────────────────────────│
│                                                                              │
│   messages[0] = {                                                            │
│       role: "system",                                                        │
│       content: build_prompt(discoveries, last_response)  ← 🔄 ŚWIEŻE DANE!   │
│   }                                                                          │
│                                                                              │
│   ┌─────────────────────────────────────────────────────────────────────┐    │
│   │  🎯 MECHANIZM UCZENIA SIĘ W TRAKCIE SESJI                           │    │
│   │     - discoveries: co już wiemy (alergie, historia...)              │    │
│   │     - last_response: co restauracja właśnie powiedziała             │    │
│   └─────────────────────────────────────────────────────────────────────┘    │
│                                                                              │
└──────────────────────────────────────────────────────────────────────────────┘
                                       │
                                       ▼
┌──────────────────────────────────────────────────────────────────────────────┐
│                    KROK B: 🧠 MYŚLENIE (NextStep)                            │
│ ─────────────────────────────────────────────────────────────────────────────│
│                                                                              │
│        ┌─────────────┐                     ┌─────────────────────┐           │
│        │   messages  │ ─────────────────▶  │     LLM API         │           │
│        │   (lista)   │                     │  response_model=    │           │
│        └─────────────┘                     │    NextStep         │           │
│                                            └──────────┬──────────┘           │
│                                                       │                      │
│                                                       ▼                      │
│   ┌─────────────────────────────────────────────────────────────────────┐    │
│   │                         step: NextStep                              │    │
│   │  ┌───────────────────────────────────────────────────────────────┐  │    │
│   │  │ current_state: "Co się wydarzyło?"        ← 💭 Ocena sytuacji │  │    │
│   │  │ plan_remaining_steps: ["krok1", "krok2"]  ← 📋 Plan (0-5)     │  │    │
│   │  │ plan_changed: true/false                  ← 🔄 Czy zmiana?    │  │    │
│   │  │ next_action: QueryCustomerData|Say|...   ← ⚡ AKCJA            │  │    │
│   │  └───────────────────────────────────────────────────────────────┘  │    │
│   └─────────────────────────────────────────────────────────────────────┘    │
│                                                                              │
└──────────────────────────────────────────────────────────────────────────────┘
                                       │
                                       ▼
┌──────────────────────────────────────────────────────────────────────────────┐
│                    KROK C: 🏁 OBSŁUGA STANÓW KOŃCOWYCH                       │
│ ─────────────────────────────────────────────────────────────────────────────│
│                                                                              │
│                         step.next_action                                     │
│                              │                                               │
│              ┌───────────────┼───────────────┐                               │
│              ▼               │               ▼                               │
│   ┌─────────────────┐        │    ┌─────────────────┐                        │
│   │    Complete?    │        │    │    Abandon?     │                        │
│   └────────┬────────┘        │    └────────┬────────┘                        │
│            │ TAK             │             │ TAK                             │
│            ▼                 │             ▼                                 │
│   ┌─────────────────┐        │    ┌─────────────────┐                        │
│   │ ✅ SUKCES!      │        │    │ ❌ PORAŻKA!     │                        │
│   │ return Complete │        │    │ return Abandon  │                        │
│   │ ══════════════  │        │    │ + reason        │                        │
│   │   🛑 EXIT       │        │    │ ══════════════  │                        │
│   └─────────────────┘        │    │   🛑 EXIT       │                        │
│                              │    └─────────────────┘                        │
│                              │                                               │
│                              ▼ NIE (żaden terminal)                          │
│                        [KONTYNUUJ]                                           │
│                                                                              │
└──────────────────────────────────────────────────────────────────────────────┘
                                       │
                                       ▼
┌──────────────────────────────────────────────────────────────────────────────┐
│                    KROK D: ⚡ WYKONANIE (Dispatch)                            │
│ ─────────────────────────────────────────────────────────────────────────────│
│                                                                              │
│   feedback = dispatch(                                                       │
│       step.next_action,     ← 🎯 Co zrobić                                   │
│       discoveries,          ← 🧠 Lista do MUTACJI!                           │
│       goal,                 ← 📋 Cel rezerwacji                              │
│       scenario              ← 🎭 Scenariusz symulacji                        │
│   )                                                                          │
│                                                                              │
│   ┌─────────────────────────────────────────────────────────────────────┐    │
│   │                    🔀 ROUTING W DISPATCH                            │    │
│   │                                                                     │    │
│   │   QueryCustomerData ──▶ Pobierz dane klienta → discoveries.append() │    │
│   │   CheckHistory      ──▶ Sprawdź historię    → discoveries.append()  │    │
│   │   Say               ──▶ Symuluj rozmowę     → feedback = odpowiedź  │    │
│   │                                                                     │    │
│   └─────────────────────────────────────────────────────────────────────┘    │
│                                                                              │
│   📥 feedback = "wynik akcji (string)"                                       │
│                                                                              │
└──────────────────────────────────────────────────────────────────────────────┘
                                       │
                                       ▼
┌──────────────────────────────────────────────────────────────────────────────┐
│                    KROK E: 🔄 PĘTLA SPRZĘŻENIA ZWROTNEGO                     │
│ ─────────────────────────────────────────────────────────────────────────────│
│                                                                              │
│   messages.append({                                                          │
│       role: "user",                                                          │
│       content: f"Wynik akcji: {feedback}"    ← 📝 Dodaj do historii          │
│   })                                                                         │
│                                                                              │
│   ┌─────────────────────────────────────────────────────────────────────┐    │
│   │  if isinstance(step.next_action, Say):                              │    │
│   │      last_response = feedback   ← 🏪 Zapamiętaj odpowiedź rest.     │    │
│   │                                    (trafi do następnego promptu!)   │    │
│   └─────────────────────────────────────────────────────────────────────┘    │
│                                                                              │
└──────────────────────────────────────────────────────────────────────────────┘
                                       │
                                       ▼
┌──────────────────────────────────────────────────────────────────────────────┐
│                    KROK F: 🪟 OPTYMALIZACJA (Sliding Window)                 │
│ ─────────────────────────────────────────────────────────────────────────────│
│                                                                              │
│   if len(messages) > 9:                                                      │
│       messages = [messages[0]] + messages[-8:]                               │
│                   ▲                    ▲                                     │
│                   │                    │                                     │
│            System Prompt         Ostatnie 8                                  │
│            (zawsze!)             wiadomości                                  │
│                                                                              │
│   ┌─────────────────────────────────────────────────────────────────────┐    │
│   │  💡 DLACZEGO?                                                       │    │
│   │     - Ograniczenie kontekstu LLM (tokeny)                           │    │
│   │     - System prompt ZAWSZE aktualny (z discoveries!)                │    │
│   │     - Najnowsze wiadomości = najważniejsze                          │    │
│   └─────────────────────────────────────────────────────────────────────┘    │
│                                                                              │
└──────────────────────────────────────────────────────────────────────────────┘
                                       │
                                       │
                    ┌──────────────────┴───────────────┐
                    │                                  │
                    ▼                                  │
          ┌─────────────────┐                          │
          │  i < 10?        │                          │
          └────────┬────────┘                          │
                   │                                   │
         ┌─────────┴─────────┐                         │
         ▼ TAK               ▼ NIE                     │
    ┌─────────┐       ┌──────────────┐                 │
    │ NEXT    │       │ ⚠️ TIMEOUT   │                 │
    │ TURA    │───────│ return None  │                 │
    │ (i+1)   │  ▲    │ Max 10 tur   │                 │
    └────┬────┘  │    └──────────────┘                 │
         │       │                                     │
         └───────┴─────────────────────────────────────┘
                        🔁 LOOP BACK TO KROK A
```                        

In [ ]:
def make_reservation(
    customer_id: str, 
    goal: ReservationGoal,
    scenario: RestaurantScenario = RestaurantScenario.HAPPY_PATH  
):
    """
    GŁÓWNA PĘTLA AGENTA
    
    To tutaj 'NextStep' spotyka się z 'dispatch'.
    """

    print(f"\n{'═'*70}")
    print(f"📞 ROZPOCZYNAM REZERWACJĘ")
    print(f"{'═'*70}")
    print(f"🏪 Restauracja: {goal.restaurant}")
    print(f"📅 Data: {goal.date}, Godzina: {goal.time}")
    print(f"👥 Liczba osób: {goal.party_size}")
    print(f"👤 Klient: {customer_id}")
    print(f"🎭 Scenariusz: {scenario.value}")
    print(f"{'═'*70}\n")

    # ───────────────────────────────────────────────────────────────────────
    # 1. Inicjalizacja Stanu Sesji
    # ───────────────────────────────────────────────────────────────────────
    
    # Wiedza odkryta - startujemy z pustą listą. 
    # To dispatch będzie ją wypełniał (mutował)!
    discoveries = []  
    
    # Historia rozmowy dla modelu
    messages = []     
    last_response = "" 

    # Budujemy pierwszy prompt (jeszcze bez odkryć)
    system_msg = {"role": "system", "content": build_prompt([], "")}
    messages.append(system_msg)

    # Definiujemy zadanie dla Agenta
    task = f"Zarezerwuj stolik w {goal.restaurant} na {goal.date} o {goal.time} dla {goal.party_size} osób. Klient: {customer_id}"
    messages.append({"role": "user", "content": task})

    # ───────────────────────────────────────────────────────────────────────
    # 2. GŁÓWNA PĘTLA (Hard limit 10 tur dla bezpieczeństwa)
    # ───────────────────────────────────────────────────────────────────────

    for i in range(10):
        print(f"\n{'─'*70}")
        print(f"🔄 TURA {i+1}/10")
        print(f"{'─'*70}")

        # ═══════════════════════════════════════════════════════════════════
        # KROK A: AKTUALIZACJA PROMPTU (Critical Path!)
        # ═══════════════════════════════════════════════════════════════════
        # Nadpisujemy messages[0] nową wersją promptu, która zawiera
        # zaktualizowane 'discoveries' i 'last_response'.
        # To jest mechanizm uczenia się w trakcie sesji.
        messages[0] = {
            "role": "system", "content": build_prompt(discoveries, last_response)
        }

        # ═══════════════════════════════════════════════════════════════════
        # KROK B: MYŚLENIE (NextStep)
        # ═══════════════════════════════════════════════════════════════════
        # LLM analizuje sytuację i zwraca strukturę decyzyjną.
        step = h.call_api(messages, response_model=NextStep)

        print(f"\n💭 OCENA SYTUACJI:\n   {step.current_state}")
        print(f"\n📋 PLAN (wykonam TYLKO pierwszy krok!):")
        for idx, plan_step in enumerate(step.plan_remaining_steps, 1):
            marker = "→" if idx == 1 else " "
            print(f"   {marker} {idx}. {plan_step}")
        print(f"\n⚡ AKCJA: {step.next_action.action}")

        # ═══════════════════════════════════════════════════════════════════
        # KROK C: OBSŁUGA STANÓW KOŃCOWYCH (Terminal States)
        # ═══════════════════════════════════════════════════════════════════
        if isinstance(step.next_action, Complete):
            print(f"\n{'═'*70}\n✅ SUKCES! REZERWACJA ZAKOŃCZONA\n{'═'*70}")
            return step.next_action

        if isinstance(step.next_action, Abandon):
            print(f"\n{'═'*70}\n❌ PORAŻKA! REZERWACJA PRZERWANA\nPOWÓD: {step.next_action.reason}\n{'═'*70}")
            return step.next_action

        # ═══════════════════════════════════════════════════════════════════
        # KROK D: WYKONANIE (Dispatch)
        # ═══════════════════════════════════════════════════════════════════
        # Tu dzieje się magia: 
        # 1. Wywołujemy narzędzie/rozmowę.
        # 2. 'discoveries' są aktualizowane in-place!
        feedback = dispatch(
            step.next_action, 
            discoveries, 
            goal,         
            scenario      
        )
        print(f"\n📥 FEEDBACK: {feedback}")

        # ═══════════════════════════════════════════════════════════════════
        # KROK E: PĘTLA SPRZĘŻENIA ZWROTNEGO
        # ═══════════════════════════════════════════════════════════════════
        # Doklejamy wynik akcji do historii, aby LLM "widział", co się stało.
        messages.append({
            "role": "user",
            "content": f"Wynik akcji: {feedback}"
        })

        # Jeśli to była rozmowa, zapisujemy odpowiedź do kontekstu promptu
        if isinstance(step.next_action, Say):
            print(f"\n🤖 MÓWIĘ: {step.next_action.utterance}")
            print(f"🏪 RESTAURACJA: {feedback}")
            last_response = feedback
            
        # ═══════════════════════════════════════════════════════════════════
        # KROK F: OPTYMALIZACJA KONTEKSTU (Sliding Window)
        # ═══════════════════════════════════════════════════════════════════
        # Zachowaj tylko System Prompt [0] + ostatnie 8 wiadomości
        if len(messages) > 9:
            messages = [messages[0]] + messages[-8:]
            
    print(f"\n⚠️  Osiągnięto maksymalną liczbę tur (10)")
    return None

### Analiza mechaniki – z lotu ptaka

Spójrzmy na ten kod z dystansu. Widzisz tu sporo konkretnych ustawień: historia przycięta do 8 ostatnich wiadomości, pętla ograniczona do 10 kroków, specyficzny sposób formatowania logów.

Czy to są jedyne słuszne wartości? Absolutnie nie. To są setki mikro-decyzji, które w prawdziwym projekcie zależałyby od budżetu, wymagań biznesowych czy specyfiki API. Na tym etapie w ogóle ich nie roztrząsamy – to są "pokrętła", którymi będziesz kręcić później, optymalizując system.

To, co jest tu absolutnie kluczowe i co musisz zrozumieć, to **przepływ informacji** między trzema głównymi elementami, które zbudowaliśmy:

1.  **Dynamiczny prompt (`messages[0] = ...`)**: To jest moment krytyczny. W każdej turze nadpisujemy instrukcję systemu. W turze 1 agent nie wie nic. W turze 2 "widzi" już w prompcie, że klient ma alergię (bo funkcja `dispatch` zaktualizowała listę `discoveries`).
2.  **NextStep (centrum decyzyjności)**: Dzięki temu, że prompt się zmienił, model generuje **nowy plan**. Nie musimy kodować ręcznie `if allergy then say...`. Model sam to wywnioskuje i zmieni strategię, bo zmieniła się jego definicja rzeczywistości.
3.  **Dispatch (ręce)**: Wykonuje ten plan, dostarcza nowe fakty i pętla się zamyka.

Teraz, mając te wszystkie elementy spięte w jedną całość, jesteśmy gotowi, by w końcu przekręcić kluczyk w stacyjce.

## Sprawdzamy

Moment prawdy. Mamy zdefiniowane wszystkie warstwy – od bazy danych po pętlę decyzyjną. Czas przekręcić kluczyk w stacyjce.

Uruchomimy teraz agenta dla Jana Kowalskiego w scenariuszu `HAPPY_PATH`. Obserwuj uważnie konsolę – zobaczysz tam nie tylko to, co agent mówi, ale przede wszystkim to, jak **myśli** i jak buduje swoją wiedzę krok po kroku.

In [ ]:
make_reservation(
    customer_id="kowalski",
    goal=ReservationGoal(
        restaurant="Bella Italia",
        date="2024-03-15",
        time="19:00",
        party_size=2
    ),
    scenario=RestaurantScenario.HAPPY_PATH 
)

### Analiza przebiegu (Trace) 

Spójrzmy na ten log. To jest dowód na to, że nasz system działa zgodnie z założeniami architektury Software 3.0.

1.  **Dyscyplina Discovery (Tury 1-2):**
    Zwróć uwagę, że Agent **nie rzucił się od razu do telefonu**. Przez pierwsze dwie tury cierpliwie "odrabiał pracę domową". Najpierw pobrał dane klienta (`query_customer`), a potem sprawdził historię (`check_history`). To nie przypadek – to efekt naszego Workflow w System Prompcie.

2.  **Efekt Pamięci (Tura 3):**
    To jest najważniejszy moment. Zobacz, co powiedział Agent:
    > *"Rezerwacja na nazwisko Kowalski. **Mam alergię na orzechy**. W przeszłości mieliśmy problem, bo **nie przygotowano stolika**..."*
    
    Skąd on to wiedział? Te informacje nie były w `ReservationGoal`. One zostały dynamicznie wstrzyknięte do promptu z listy `discoveries`. To dowód na to, że pętla adaptacji działa.

3.  **Brutalna skuteczność (One-shot):**
    Możesz zauważyć, że Agent "wyrzucił" z siebie wszystkie informacje naraz. Wymienił datę, alergię i historyczny problem w jednym bloku tekstu. Z ludzkiego punktu widzenia brzmi to nieco robotycznie, ale z inżynierskiego – jest to maksymalnie skuteczne (oszczędza tury i tokeny). Restauracja (symulator) zrozumiała intencję i potwierdziła rezerwację.

4.  **Czyste lądowanie (Tura 4):**
    Gdy Agent otrzymał potwierdzenie ("Doskonale!"), nie kontynuował rozmowy na siłę. Router wybrał akcję `Complete` i zwrócił ustrukturyzowane podsumowanie.

System zadziałał idealnie w warunkach laboratoryjnych. Ale co się stanie, gdy restauracja zacznie stawiać opór? Tym zajmiemy się w kolejnych testach.

## Podsumowanie

Gratulacje. Właśnie przekroczyliśmy magiczną barierę między "rozmową z botem" a "systemem agentowym". To, co stworzyliśmy, to fundament autonomii.

Zwróć uwagę na trzy krytyczne rzeczy, które się tu wydarzyły:

1.  **Stabilizująca rola `dispatch`**: To jest najważniejszy element inżynierski tej lekcji. Zauważ, że niezależnie od tego, jak bardzo kreatywny (lub szalony) jest model w wymyślaniu planów, funkcja `dispatch` sprowadza wszystko do parteru. To ona decyduje, co faktycznie wolno zrobić. To jest ten bezpiecznik, który pozwala nam spać spokojnie.
2.  **Dynamiczna natura `NextStep`**: Być może zauważasz już, że nasza struktura `NextStep` w połączeniu z pętlą główną daje niesamowity efekt. Agent nie realizuje ślepo planu ułożonego na początku (jak w klasycznym programowaniu). On w każdej iteracji, co kilka sekund, **rewaluuje rzeczywistość**. Jeśli w kroku 3 dowie się czegoś, co podważa sens kroku 4 – natychmiast zmieni strategię. To jest definicja adaptacji.
3.  **Czystość danych**: Dzięki mechanizmowi `discoveries` oddzieliliśmy "fakty" od "szumu rozmowy". To pozwala utrzymać higienę w kontekście i sprawia, że agent podejmuje decyzje na podstawie twardych danych, a nie domysłów.

### Co dalej?
Już zrobiliśmy pewne kroki do przodu, ale to jeszcze nie wszystko. W następnej lekcji zrobimy jeszcze kilka kroków do przodu.